# Notebook 04 - BERT y Aplicaciones

## Objetivos
- Clasificar sentimiento con DistilBERT fine-tuned.
- Responder preguntas con pipeline de QA.
- Extraer entidades (NER) y embeddings contextuales.

## Introduccion
BERT es encoder-only: excelente para entender texto. Exploraremos sentimiento, QA, NER y vectores `[CLS]` sobre datasets del curso.

In [1]:
from pathlib import Path
from IPython.display import display
import pandas as pd
import torch
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModel,
)

RUTA_REVIEWS = Path('..') / 'datasets' / 'reviews_sentiment.csv'
RUTA_QA = Path('..') / 'datasets' / 'documentos_qa.csv'
print('Archivos listos:', RUTA_REVIEWS.exists(), RUTA_QA.exists())

c:\Users\juand\anaconda3\envs\tf_windows\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Archivos listos: True True


## 1) Sentimiento con modelo fine-tuned SST-2

In [2]:
# Pipeline de clasificacion de sentimiento en ingles
sentiment = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
)

df_reviews = pd.read_csv(RUTA_REVIEWS)
display(df_reviews.head())

c:\Users\juand\anaconda3\envs\tf_windows\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\juand\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. 

,texto,sentimiento
0,"Excelente producto, llegó rápido y funciona pe...",positivo
1,"Muy mala calidad, se rompió al segundo día.",negativo
2,El servicio al cliente fue amable y resolvió m...,positivo
3,Demasiado caro para lo que ofrece.,negativo
4,La interfaz es intuitiva y fácil de usar.,positivo


In [3]:
# Traducimos etiquetas al ingles para el modelo SST-2
mapa_en = {
    'Excelente producto, llegó rápido y funciona perfectamente.': 'Excellent product, arrived fast and works perfectly.',
    'Muy mala calidad, se rompió al segundo día.': 'Very bad quality, broke on the second day.',
    'El servicio al cliente fue amable y resolvió mi problema.': 'Customer service was friendly and solved my problem.',
    'Demasiado caro para lo que ofrece.': 'Too expensive for what it offers.',
    'La interfaz es intuitiva y fácil de usar.': 'The interface is intuitive and easy to use.',
}

resultados = []
for _, row in df_reviews.iterrows():
    texto_es = row['texto']
    texto_en = mapa_en.get(texto_es, texto_es)
    pred = sentiment(texto_en[:512])[0]
    resultados.append({
        'texto': texto_es,
        'sentimiento_real': row['sentimiento'],
        'label_modelo': pred['label'],
        'score': round(pred['score'], 4),
    })

df_sent = pd.DataFrame(resultados)
display(df_sent)

,texto,sentimiento_real,label_modelo,score
0,"Excelente producto, llegó rápido y funciona pe...",positivo,POSITIVE,0.9999
1,"Muy mala calidad, se rompió al segundo día.",negativo,NEGATIVE,0.9998
2,El servicio al cliente fue amable y resolvió m...,positivo,POSITIVE,0.9866
3,Demasiado caro para lo que ofrece.,negativo,NEGATIVE,0.9998
4,La interfaz es intuitiva y fácil de usar.,positivo,POSITIVE,0.9997
5,"No recomiendo esta marca, muy decepcionante.",negativo,NEGATIVE,0.9890
6,Entrega puntual y empaque impecable.,positivo,NEGATIVE,0.9694
7,El producto no coincide con la descripción.,negativo,NEGATIVE,0.9801
8,"Superó mis expectativas, lo volvería a comprar.",positivo,NEGATIVE,0.9949
9,Atención telefónica lenta y poco útil.,negativo,NEGATIVE,0.9812


## 2) Question Answering sobre documentos

In [4]:
# Pipeline extractivo de respuestas
qa = pipeline('question-answering', model='distilbert-base-cased-distilled-squad')
df_qa = pd.read_csv(RUTA_QA)
display(df_qa.head())

c:\Users\juand\anaconda3\envs\tf_windows\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\juand\.cache\huggingface\hub\models--distilbert-base-cased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling ba

,contexto,pregunta,respuesta
0,Python fue creado por Guido van Rossum y publi...,¿Quién creó Python?,Guido van Rossum
1,Python fue creado por Guido van Rossum y publi...,¿En qué año se publicó Python?,1991
2,Los Transformers fueron introducidos en el pap...,¿Qué empresa introdujo los Transformers?,Google
3,Los Transformers fueron introducidos en el pap...,¿En qué año se publicó el paper?,2017
4,BERT es un modelo encoder-only entrenado con M...,¿Qué tipo de arquitectura usa BERT?,encoder-only


In [5]:
respuestas = []
for _, row in df_qa.iterrows():
    out = qa(question=row['pregunta'], context=row['contexto'])
    respuestas.append({
        'pregunta': row['pregunta'],
        'respuesta_esperada': row['respuesta'],
        'respuesta_modelo': out['answer'],
        'score': round(out['score'], 4),
    })

display(pd.DataFrame(respuestas))

,pregunta,respuesta_esperada,respuesta_modelo,score
0,¿Quién creó Python?,Guido van Rossum,Python fue creado por Guido van Rossum y publi...,0.1279
1,¿En qué año se publicó Python?,1991,1991,0.1234
2,¿Qué empresa introdujo los Transformers?,Google,Google,0.0967
3,¿En qué año se publicó el paper?,2017,Los Transformers fueron introducidos,0.6124
4,¿Qué tipo de arquitectura usa BERT?,encoder-only,con Masked Language Modeling y Next Sentence P...,0.0386
5,¿Qué predice GPT?,la siguiente palabra,only,0.0328
6,¿Qué permite la Self-Attention?,que cada token observe a todos los demás tokens,La capa de Self-Attention permite que cada token,0.3655
7,¿Cómo reformula T5 las tareas?,texto a texto,NLP como problemas de texto a texto,0.1735


## 3) Named Entity Recognition (NER)

In [6]:
# Pipeline NER en ingles
ner = pipeline('ner', grouped_entities=True)
texto_ner = 'Guido van Rossum created Python at Google Brain in 1991.'
entidades = ner(texto_ner)
display(pd.DataFrame(entidades))

No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496 (https://huggingface.co/dbmdz/bert-large-cased-finetuned-conll03-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
c:\Users\juand\anaconda3\envs\tf_windows\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\juand\.cache\huggingface\hub\models--dbmdz--bert-large-cased-finetuned-conll03-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Pyt

,entity_group,score,word,start,end
0,PER,0.998653,Guido van Rossum,0,16
1,MISC,0.907441,Python,25,31
2,MISC,0.786238,Google Brain,35,47


## 4) Extraccion de embeddings contextuales

In [7]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModel.from_pretrained('distilbert-base-uncased')
model.eval()

frase = 'Transformers changed natural language processing'
inputs = tokenizer(frase, return_tensors='pt')
with torch.no_grad():
    hidden = model(**inputs).last_hidden_state

# Vector CLS (primer token)
cls_vector = hidden[0, 0, :]
print('Dimension embedding CLS:', cls_vector.shape)
print('Primeros 5 valores:', cls_vector[:5].tolist())

Dimension embedding CLS: torch.Size([768])
Primeros 5 valores: [-0.40909767150878906, -0.22725710272789001, -0.0709218829870224, -0.10708536207675934, -0.02807406708598137]


## Resultados
Clasificamos reseñas, respondimos preguntas extractivas, detectamos entidades y extrajimos un embedding `[CLS]` contextual.

## Conclusiones
BERT destaca en comprension y tareas de entendimiento. Para produccion conviene validar idioma, calibrar umbrales y considerar modelos multilingues.

## Ejercicios guiados resueltos
**Ejercicio:** Calcula accuracy aproximada mapeando POSITIVE->positivo.

**Solucion:**

In [8]:
df_sent['pred_es'] = df_sent['label_modelo'].map({'POSITIVE': 'positivo', 'NEGATIVE': 'negativo'})
acc = (df_sent['pred_es'] == df_sent['sentimiento_real']).mean()
print(f'Accuracy aproximada: {acc:.2%}')

Accuracy aproximada: 75.00%


## Ejercicios propuestos
1. Prueba un modelo multilingue para reseñas en espanol.
2. Fine-tune ligero de sentimiento con `Trainer`.
3. Compara similitud coseno entre embeddings de dos frases.

## Preguntas de reflexion
1. Cuando falla QA extractivo aun con contexto correcto?
2. Que diferencia hay entre NER pipeline y token classification?
3. Por que `[CLS]` resume la secuencia en BERT?